#### In this part, the model related tasks will be performed. To achive the goal, we will be having the collaborative and content-base approach where i'll be using  scikit-learns's surprise and custom build matrix. Alternatively,Lightfm(for python3.9 as it has some build issue), Implicit, Cosine simillarity can be used to achive the exact goal.

In [20]:
#following are the necessary libraries to be imported
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
import pickle

#up-untill-now to use the surprise library we need to install numpy 1.26.4 or something compatible

In [21]:
cleaned_df = pd.read_csv("E:\Coding\ML_projects\Movie-Recommendation-System\data\processed\cleaned_data_final.csv")
cleaned_df.head()

,userId,movieId,title,release_year_scaler,rating_scaler,(no genres listed),Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1,Toy Story,0.801724,0.777778,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,3,Grumpier Old Men,0.801724,0.777778,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,1,6,Heat,0.801724,0.777778,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1,47,Seven (a.k.a. Se7en),0.801724,1.000000,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,1,50,"Usual Suspects, The",0.801724,1.000000,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


### In this section: will train the SVD model for the colleborative filtering of our recommendation system

In [22]:
#Reader object tells 'surprise' how to interpret rating values from the dataFrame and we have a range of 0 to 1 as we have scaled the ratings.
reader = Reader(rating_scale=(0, 1))
#this will convert the cleaned dataset's userid,movieid,rating_sclaer portion into surprise’s internal dataset format.
#this will help the library to manage indexing,validation, and train/test splits
data = Dataset.load_from_df(cleaned_df[['userId', 'movieId', 'rating_scaler']], reader)
data.raw_ratings[:5]  #displays first 5 raw ratings from the dataset and donot panic about the none as it is for timestamp which we are not using

[(1, 1, 0.7777777777777777, None),
 (1, 3, 0.7777777777777777, None),
 (1, 6, 0.7777777777777777, None),
 (1, 47, 1.0, None),
 (1, 50, 1.0, None)]

In [23]:
#converted the full dataset(no train test split) into the Trainset object so we can produce a model trained on all available data for deployment
trainset = data.build_full_trainset()
model = SVD()
model.fit(trainset)
#traininf of the (SVD)-a matrix-factorization model.
#model learns hidden features for users and movies plus biases, enabling accurate scoring of unseen user–item pairs

In [24]:
#Now lets save the trained model using pickle to use it later for making predictions
with open('movie_recom_svd_model.pkl', 'wb') as f:
    pickle.dump(model, f)

### Preparation of Genre-Based Filtering (Content-Based)

#### To do so, we'll need to have a genre-movie dataframe and a title dataframe

In [25]:
#will extract one-hot encoded genres for content-based filtering
movie_genres_df=cleaned_df.drop(columns=['userId', 'movieId', 'rating_scaler', 'release_year_scaler','title'])
movie_genres_df.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0


In [26]:
with open('movie_genres.pkl', 'wb') as f:
    pickle.dump(movie_genres_df.values, f)
#saving the genres one-hot encoded matrix for later use

In [27]:
#for movie titles dataframe
with open('movie_titles.pkl', 'wb') as f:
    pickle.dump(cleaned_df['title'].values, f)

### Now, our model training and needed data matrix are done and saved for the Fastapi use. Let's see the fastAPI part. FastAPI basic is required for this.